# Cross-shelf wave paper: Colab pipeline (step by step)

Ei notebook tomar thesis-er Drive-e thaka processed data theke shuru kore baseline, EVT, LSTM, storm evaluation, table ar figure porjonto cholbe.
**Kono notun big download lagbe na.** Notun jinish Drive-e shudhu `wave_paper/outputs` folder-e save hobe.

| Step | Ki hoy | Runtime | Somoy (anuman) |
|---|---|---|---|
| 1-3 | Drive mount, code unzip, test | CPU | 5 min |
| 4-5 | Thesis parquet -> station CSV, data audit | CPU | 5 min |
| 6 | Baseline (persistence, ridge, XGBoost) | CPU | 15-30 min |
| 7-8 | IBTrACS, storm list, EVT | CPU | 15-30 min |
| 9-10 | LSTM training (GPU) | **GPU (T4)** | Step 9-e measure koro |
| 11-12 | Table, storm result, figure, zip | CPU | 10 min |

**Storage:** nijer notun output ~0.3-1 GB. Tomar 5-10 GB limit-er onek niche. Raw ERA5/CMEMS NetCDF file notun kore copy kora hoy na.

**Session disconnect hole:** Step 1 ar 2 abar chalao, tarpor jei step-e chhile oi step theke cholo. Shob result Drive-e save hoy, ar LSTM training restart-safe.

## Step 1: Google Drive mount ar path thik kora

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess, zipfile
import pandas as pd

DRIVE = pathlib.Path('/content/drive/MyDrive')

# --- tomar thesis-er existing file ---
THESIS_PROCESSED = DRIVE / 'wave_thesis' / 'data' / 'processed'   # P1_merged_raw.parquet ... P5_merged_raw.parquet
WAVE_NC = DRIVE / 'wave_cmems_nbob.nc'                            # optional: shudhu grid-cell coordinate ber korar jonno
WIND_NC = DRIVE / 'era5_yearly_backup' / 'wind_nbob_1993.nc'      # optional: same karone

# --- ei paper-er notun folder ---
PAPER = DRIVE / 'wave_paper'
OUT = PAPER / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

for p in [THESIS_PROCESSED, WAVE_NC, WIND_NC, PAPER / 'cross-shelf-wave-hs-bay-of-bengal.zip']:
    print('OK     ' if p.exists() else 'MISSING', p)

**Ki dekhbe:** 4 ta line-i `OK` hote hobe. `MISSING` dekhale:
- `...zip` MISSING hole: ami je zip file dichhi seta Drive-er `wave_paper` folder-e upload koro (folder na thakle banao).
- `THESIS_PROCESSED` MISSING hole: Drive-e `P1_merged_raw.parquet` file-gulo kothay ache dekhe path-ta thik koro.
- `WAVE_NC` / `WIND_NC` MISSING hole kono somoshya nei, shudhu Step 5-er grid-cell part skip hobe.

## Step 2: Code unzip kora (Colab-er local disk-e) ar package install

In [ ]:
WORK = pathlib.Path('/content/work'); WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(PAPER / 'cross-shelf-wave-hs-bay-of-bengal.zip') as z:
    z.extractall(WORK)
REPO = WORK / 'cross-shelf-wave-hs-bay-of-bengal'
os.chdir(REPO)

!pip -q install pytest xarray netCDF4 pyarrow
DATA = str(OUT / 'data' / 'stations')   # station CSV Drive-e save hobe
RES = str(OUT)
print('working in', os.getcwd())

## Step 3: Code test (optional kintu recommended, ~20 second)

In [ ]:
!python -m pytest -q

**Ki dekhbe:** `passed` (1 ta `skipped` thakle thik achhe, eta PyTorch test). `failed` dekhale full message amake pathao.

## Step 4: Thesis-er parquet file -> station CSV

In [ ]:
!python scripts/convert_thesis_parquet.py --src "{THESIS_PROCESSED}" --out "{DATA}"

**Ki dekhbe:** prottek station-er line-e `missing steps 0` ar `NaN values 0`, date range `1993-01-01 ... 2023-12-31`.
Jodi `missing steps` > 0 hoy tahole amake bolo, kaj thamiye age eta thik korte hobe.

## Step 5: Data audit (descriptive stats, grid cell)

In [ ]:
EXTRA = f'--wave-nc "{WAVE_NC}" --wind-nc "{WIND_NC}"' if WAVE_NC.exists() and WIND_NC.exists() else ''
!python scripts/audit_stations.py --data "{DATA}" --out "{RES}/tables" {EXTRA}

Output: `outputs/tables/station_descriptive_stats.csv` (paper-er Table 1/Discussion-er jonno: mean, P90, P99, max Hs, Tp, wind) ar `grid_cells.csv` (je grid cell asholei use hoyeche).
**Tomar kaj:** GEBCO viewer-e (download.gebco.net) prottek station-er depth dekhe likhe rakho (P1-P5), ar `grid_cells.csv` save rakho.

## Step 6: Baseline forecast (persistence, ridge, XGBoost)
CPU-te cholbe, XGBoost-er jonno ~15-30 minute lagte pare. Jodi time kom, `--skip-xgb` jog koro (kintu paper-e XGBoost lagbe).

In [ ]:
!python scripts/run_baselines.py --data "{DATA}" --out "{RES}/baselines"

**Ki dekhbe:** 15 ta `... h done` line. Output: `outputs/baselines/baseline_metrics.csv`.

## Step 7: IBTrACS cyclone track ar storm list
Internet theke NOAA-r file nibe (~ kichu MB). Download fail korle error-e je link ache seta browser-e khule file nijer hate download kore `--csv` diye path dao.

In [ ]:
!python scripts/make_ibtracs_subset.py --out "{RES}/ibtracs_nio.csv"
!python scripts/list_storms.py --tracks "{RES}/ibtracs_nio.csv" --from 2019 --to 2023 --max-km 800 --out "{RES}/tables/storms_listed.csv"

**Ki dekhbe:** 2019-2023-er je cyclone gulo station-er 800 km-er moddhe eshechhe tar naam, tarikh, ar P1-P5 theke sobcheye kachhe durotto (km).
Nicher cell-e storm window gulo list-er shathe milie thik koro (start/end date). Amphan ar Mocha-r window thesis theke neya, baki gulo anumanik.

In [ ]:
import yaml
STORMS = {   # naam: [start, end]   <-- storms_listed.csv er shathe milao
    'Fani':    ['2019-04-26', '2019-05-04'],
    'Bulbul':  ['2019-11-05', '2019-11-11'],
    'Amphan':  ['2020-05-18', '2020-05-22'],
    'Yaas':    ['2021-05-23', '2021-05-27'],
    'Sitrang': ['2022-10-22', '2022-10-26'],
    'Mocha':   ['2023-05-12', '2023-05-16'],
    'Midhili': ['2023-11-15', '2023-11-18'],
}
cfg_path = REPO / 'config' / 'params.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['storms_in_test_period'] = STORMS
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print('storm windows saved:', list(STORMS))

## Step 8: Extreme-value analysis (EVT)
Main run (800 km, 48 h) ar cyclone-distance sensitivity (500 km, 1000 km). ~10-20 minute.

In [ ]:
!python scripts/run_evt.py --data "{DATA}" --out "{RES}/evt" --tracks "{RES}/ibtracs_nio.csv"
for km in (500, 1000):
    subprocess.run(f'python scripts/run_evt.py --data "{DATA}" --out "{RES}/evt_km{km}" --tracks "{RES}/ibtracs_nio.csv" --cyclone-km {km}', shell=True, check=True)
pd.read_csv(f'{RES}/evt/evt_summary.csv').set_index('station').T

**Ki dekhbe (`evt_summary` table):**
- `xi`, `xi_lo`, `xi_hi`: interval 0-ke cover korle paper-e "bounded tail" likha jabe na.
- `rl100` (pooled), `rl100_combined` (cyclone + others), `gev_rl100` (annual maxima): tinta kachakachi hole bhalo.
- `cyclone_fit_unreliable = True` hole cyclone storm-er shonkha kom, amake bolo.
- `pettitt_year`, `pettitt_p`: reanalysis series-e break ache kina (2017 kache hole sondeho).
- Thresholds: `evt/P1_mrl.csv`, `evt/P1_stability.csv` figure-e dekha jabe (Step 11).

## Step 9: GPU check ar time-estimate
**Runtime -> Change runtime type -> T4 GPU** select kore Step 1-2 abar chalao (session restart hole), tarpor eta.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ekta chhoto run diye somoy mape (P1, 24 h, Huber, seed 0)
!python scripts/run_lstm.py --data "{DATA}" --out "{RES}/lstm_timing" --stations P1 --horizons 24 --losses huber --seeds 0
t = pd.read_csv(f'{RES}/lstm_timing/lstm_metrics.csv')['minutes'].iloc[0]
print(f'one run takes {t} min')
for name, runs in {'quick (135 runs)': 135, 'core (375 runs)': 375, 'full (900 runs)': 900}.items():
    print(f'{name}: about {runs * t / 60:.1f} hours')

**Decision:** free Colab-e ek session ~ kichu ghonta cholte pare. `quick` (3 loss x 3 seed) minimum, `core` (5 loss x 5 seed) paper-er jonno bhalo, `full` (6 loss x 10 seed) best.
Somoy beshi hole `core` niye shuru koro, pore bakita jog korte parbe (nicher command same, shudhu `--preset` ba `--seeds` bodlao).

## Step 10: LSTM training (restart-safe, 80 minute-er chunk-e)

In [ ]:
!python scripts/run_lstm.py --data "{DATA}" --out "{RES}/lstm" --preset core --time-budget-min 80

Ei cell **bar bar chalao** jotokkhon na `time budget reached` er bodole shob run shesh howar message ashe (ba `runs planned, N already finished` e N = total hoy).
Disconnect hole Step 1-2 chalao, tarpor ei cell abar. Ager finished run gulo skip hoy.

In [ ]:
done = len(pd.read_csv(f'{RES}/lstm/lstm_metrics.csv'))
print(f'{done} runs finished (core preset = 375)')

## Step 11: Table, storm result, figure
Training shesh hole (ba kichu run hole-o test kore dekhte paro).

In [ ]:
!python scripts/storm_eval.py --baselines "{RES}/baselines" --lstm "{RES}/lstm" --out "{RES}/storms"
!python scripts/aggregate_forecast.py --baselines "{RES}/baselines" --lstm "{RES}/lstm" --out "{RES}/tables"
!python scripts/make_figures.py --results "{RES}" --out "{RES}/figures"

Output:
- `tables/table2_forecast_skill.csv` (Table 2: seed mean +- SD, baseline soho)
- `tables/loss_pairs.csv`, `loss_pair_counts.csv` (Huber vs onnano loss, paired test)
- `tables/vs_baselines.csv` (LSTM vs persistence/ridge/XGBoost, skill score)
- `storms/storm_table3.csv` (Table 3)
- `figures/Figure_2,3,4,5,7 (.pdf ar .png)`
Figure 1 (map) ar Figure 6 (altimetry) alada, amake altimetry match-up result pathale ami banabo.

## Step 12: Ami-ke pathanor jonno zip banano (chhoto, prediction file bad)

In [ ]:
shutil.copytree(REPO / 'config', OUT / 'config', dirs_exist_ok=True)   # kon setting-e chalano hoyeche tar record
dst = PAPER / 'results_for_review.zip'
skip = {'predictions', 'history', 'stations', 'lstm_timing'}
with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in pathlib.Path(OUT).rglob('*'):
        if f.is_file() and not (skip & set(f.parts)) and f.suffix != '.nc':
            z.write(f, f.relative_to(OUT))
print(dst, round(dst.stat().st_size / 1e6, 1), 'MB')

`wave_paper/results_for_review.zip` file-ta chat-e upload koro, sathe dataset ID, version, download date ar GEBCO depth (P1-P5) likhe dao.

## Storage check

In [ ]:
!du -sh "{RES}"/* | sort -h
!df -h /content/drive | tail -1

# Appendix: Colab GPU quota shesh hole (recovery plan)

Free Colab-er GPU limit shesh hole kichu ghonta theke ~1 din GPU paoa jay na. **Tomar kora run gulo Drive-e safe.** Ager kaj nosto hoy na.
Nicher cell gulo **CPU runtime-e-o** cholbe (GPU lage na), tai ekhon-i chalate paro. Age Step 1 ar 2 chalao (notun zip Drive-e replace korar por), ar Step 7-er `STORMS` cell ekbar abar chalao.

### A1. Ki ki run shesh hoyeche dekho

In [ ]:
!python scripts/lstm_status.py --lstm "{RES}/lstm" --seeds 0 1 2 3 4

Prottek station x horizon x loss-er pashe `n/5` dekhabe (n = koyta seed shesh). Nicher "breadth-first" plan-er jonno `--seeds 0 1 2` diye abar chalate paro.
**Plan (amar suggestion):** age shob station-e ***3 seed*** shesh koro (P1, P2 te 5 seed ache, baki station-e 3 hole-i uniform 3 seed-e paper-e report kora jabe), tarpor GPU pele bakita 5 seed-e bartiye nao.

In [ ]:
!python scripts/lstm_status.py --lstm "{RES}/lstm" --seeds 0 1 2

### A2. Option 1: GPU ferot ashle (kono khoroch nei)
Quota normally 12-24 ghontar moddhe ferot ashe, tobe guarantee nei. GPU runtime nao, Step 1-2 chalao, tarpor:
- shudhu ***GPU lagbe emon cell*** (Step 9-10) GPU-te chalao, baki shob CPU-te.
- Prothome 3 seed shesh koro (ei cell ager shesh hoyar run gulo skip kore):

In [ ]:
!python scripts/run_lstm.py --data "{DATA}" --out "{RES}/lstm" --losses mse huber focal_mse focal_mse_norm target_weighted_mse --seeds 0 1 2 --time-budget-min 80

### A3. Option 2: Kaggle (free, prottek shoptahe ~30 GPU ghonta)
1. kaggle.com-e account kholo, **phone verify** koro (GPU-r jonno lage).
2. **Datasets -> New Dataset**: duita dataset upload koro: (a) `cross-shelf-wave-hs-bay-of-bengal.zip`, (b) Drive-er `wave_paper/outputs/data/stations/` folder-er 5 ta CSV (P1.csv ... P5.csv).
3. **Code -> New Notebook**, right panel-e *Add Input* diye dui dataset add koro, *Session options -> Accelerator -> GPU T4 x2* (ba P100).
4. Nicher code cell-ta Kaggle notebook-e chalao (path `!ls /kaggle/input/*` diye milie nao):
```
!ls /kaggle/input/*
!cp -r /kaggle/input/<repo-dataset>/cross-shelf-wave-hs-bay-of-bengal /kaggle/working/repo
%cd /kaggle/working/repo
!python scripts/run_lstm.py --data /kaggle/input/<stations-dataset> --out /kaggle/working/lstm_part2 --stations P3 P4 P5 --losses mse huber focal_mse focal_mse_norm target_weighted_mse --seeds 0 1 2 --time-budget-min 600
```
5. Shesh hole *Output* tab theke `lstm_part2` folder zip kore download koro, Drive-er `wave_paper/outputs/` e upload kore unzip koro, tarpor nicher merge cell chalao.

In [ ]:
# Kaggle (ba onno jaygay) theke ana lstm_part2 ke main lstm folder-e merge kora
import pandas as pd, shutil, pathlib
part = pathlib.Path(f'{RES}/lstm_part2'); main = pathlib.Path(f'{RES}/lstm')
if part.exists():
    a = pd.read_csv(main / 'lstm_metrics.csv'); b = pd.read_csv(part / 'lstm_metrics.csv')
    merged = pd.concat([a, b]).drop_duplicates(['station', 'horizon_h', 'loss', 'seed'], keep='first')
    for sub in ('predictions', 'history'):
        (main / sub).mkdir(exist_ok=True)
        for f in (part / sub).glob('*'):
            if not (main / sub / f.name).exists():
                shutil.copy(f, main / sub / f.name)
    merged.to_csv(main / 'lstm_metrics.csv', index=False)
    print(len(a), '+', len(b), '->', len(merged), 'runs')
else:
    print('lstm_part2 folder nei')

### A4. Option 3: Colab Pro / compute units (poisha diye)
Google Colab-e paid compute unit kena jay. Baki ~130 run-er jonno kichu ghonta lagbe, tai chhoto package-i jothesto, kintu eta tomar upor.

### A5. CPU-te LSTM chalano **recommend kori na**
Ei model GPU chhara onek dhire cholbe. Jodi ekdom option na thake, tahole prothome 1 ta run CPU-te chaliye minute mape dekho (Step 9-er cell), 15 min-er beshi hole baki run gulo-r jonno Kaggle ba GPU-r opekkha koro.

### A6. Kaj atkabe na: ja ekhon-i CPU-te korte paro
Step 8 (EVT) shesh na hole sheta chalao, Step 6 (baseline) shesh na hole sheta chalao. Tarpor partial result-e ***temporary*** table dekhte paro (nicher cell), jate GPU na thakleo kaj egoy:

In [ ]:
SEEDS = '0 1 2'     # jei seed set shob station-e ache seta dao
!python scripts/storm_eval.py --baselines "{RES}/baselines" --lstm "{RES}/lstm" --out "{RES}/storms" --seeds {SEEDS}
!python scripts/aggregate_forecast.py --baselines "{RES}/baselines" --lstm "{RES}/lstm" --out "{RES}/tables" --seeds {SEEDS}
!python scripts/make_figures.py --results "{RES}" --out "{RES}/figures" --seeds {SEEDS}

**Sotorkota:** ei table-e jei station-e LSTM run shesh hoyni, ota faka thakbe ba kom run diye hobe; eta draft-er jonno, final-er jonno na. Final-er jonno shob station-e uniform seed set lagbe.